# ROUND 1

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt
from torchvision import datasets, transforms
from rich import print as pr

In [ ]:
torch.manual_seed(0)

tfm = transforms.ToTensor()

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

loader = torch.utils.data.DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=len(test_ds),  shuffle=False)

train_images, train_labels = next(iter(loader))
test_images,  test_labels  = next(iter(test_loader))

In [ ]:
X = train_images.view(len(train_images), -1).float()   
mean = X.mean(dim=0, keepdim=True)
Xc = X - mean

U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)   

## Statistically Independent Princinple Components

In [ ]:
torch.manual_seed(0)

k = 1
num_samples = 2

z = Xc @ Vt[:k].T  
z_mean = z.mean(0)
z_std = z.std(0)
z_new = torch.randn(num_samples, k) * z_std + z_mean

print(z_new)

X_new = z_new @ Vt[:k] + mean 
X_new = torch.clamp(X_new, 0, 1)

fig, axes = plt.subplots(1,num_samples, figsize=(num_samples*2,2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
  ax.imshow(X_new[i].view(28,28))
  ax.axis('off')
plt.tight_layout()
plt.show()

## COVARIANCE METHOD

In [ ]:
torch.manual_seed(0)

k = 4
num_samples = 5

Z = Xc @ Vt[:k].T
z_mean = Z.mean(dim=0)                     
Z_centered = Z - z_mean
cov = (Z_centered.T @ Z_centered) / (Z_centered.shape[0] - 1)   
dist = torch.distributions.MultivariateNormal(z_mean, covariance_matrix=cov)
z_new = dist.sample((num_samples,))          

X_new = z_new @ Vt[:k] + mean                  
X_new = torch.clamp(X_new, 0, 1)

fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
    ax.imshow(X_new[i].view(28, 28).detach().numpy())
    ax.axis('off')
plt.tight_layout()
plt.show()


## Principle components probing

In [ ]:
torch.manual_seed(1)

k = 2
num_samples = 10


N = 15
M = 15
anchor = 7
vals1 = torch.linspace(-anchor, anchor, N)   
vals2 = torch.linspace(-anchor, anchor, M)   
Z1, Z2 = torch.meshgrid(vals1, vals2, indexing="xy")
Z = torch.stack([Z1, Z2], dim=2)

Z_flat = Z.reshape(-1,2)
X_new = Z_flat @ Vt[:k] + mean
X_new = torch.clamp(X_new, 0, 1)
X_new = X_new.reshape(M, N, -1)

fig, axes = plt.subplots(M,N,figsize=(N,M))
for i in range(M):
    for j in range(N):
        axes[i, j].imshow(X_new[i,j].view(28,28))
        axes[i, j].axis('off')
        
        z1, z2 = Z[i, j].tolist()
        axes[i,j].set_title(f"{z1:.1f},    {z2:.1f}", fontsize=6, pad=2)
plt.show()





# ROUND 2

In [ ]:
import torch
from matplotlib import pyplot as plt
from torchvision import datasets, transforms
from rich import print as prin

In [ ]:
torch.manual_seed(0)

tfm = transforms.ToTensor()

fasion = False
dataset = datasets.MNIST if not fasion else datasets.FashionMNIST
train_dataset = dataset(root="./data", download=True, train=True, transform=tfm)
test_dataset = datasets.MNIST(root="./data", download=True, train=False, transform=tfm)

loader = torch.utils.data.DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

train_images, train_labels = next(iter(loader))
test_images, test_labels = next(iter(test_loader))

test = {}
counter = 0
while len(test) < 10:
  class_label = train_labels[counter].item()
  counter += 1
  if class_label in test:
    continue
  test[class_label] = counter - 1
print(test)


fig, axes = plt.subplots(1,10,figsize=(10,1))
for i, ax in enumerate(axes):
  ax.imshow(train_images[test[i]].squeeze(0), cmap="gray")
  ax.set_title(f"{train_labels[test[i]]}")
  ax.axis("off")
plt.show()




In [ ]:
X = train_images.view(len(train_dataset), -1)
X_mean = X.mean(0)
Xc = X - X_mean
U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)
print(Xc.shape)
print(Vt.shape)

In [ ]:
seed = 1
k = 300
num_samples = 4

Z = Xc @ Vt[:k].T
Z_mean, Z_std = Z.mean(0), Z.std(0)
Zc = Z - Z_mean
Z_cov = (Zc.T @ Zc) / (Zc.shape[0] - 1)

torch.manual_seed(seed)
Zi_sample = torch.randn(num_samples, k) * Z_std + Z_mean
X_zi_syn = torch.clamp(Zi_sample @ Vt[:k] + X_mean, 0, 1)

fig, axes = plt.subplots(1,num_samples,figsize=(num_samples*2, 2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
  ax.matshow(X_zi_syn[i].view(28,28))
  ax.axis('off')
plt.show()



torch.manual_seed(seed)
Zd_sample = torch.distributions.MultivariateNormal(Z_mean, covariance_matrix=Z_cov).sample((num_samples,))
X_zd_syn = torch.clamp(Zd_sample @ Vt[:k] + X_mean, 0, 1)

fig, axes = plt.subplots(1,num_samples,figsize=(num_samples*2, 2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
  ax.matshow(X_zd_syn[i].view(28,28))
  ax.axis('off')
plt.show()




In [ ]:
torch.manual_seed(seed)

M = 10
N = 10
anchor = 5
vals1 = torch.linspace(-anchor, anchor, M)
vals2 = torch.linspace(-anchor, anchor, N)

grid1, grid2 = torch.meshgrid(vals1, vals2, indexing="xy")
grid = torch.stack([grid1, grid2], dim=2)


grid_flat = grid.reshape(-1, 2)
grid_transformed = torch.clamp(grid_flat @ Vt[:2] + X_mean, 0, 1).reshape(M, N, -1)

fig, axes = plt.subplots(M, N, figsize=(M,N))
for i in range(M):
  for j in range(N):
    axes[i, j].imshow(grid_transformed[i,j].view(28,28))
    axes[i,j].axis('off')
    z1, z2 = Z[i, j].tolist()
    axes[i,j].set_title(f"{z1:.1f}, {z2:.1f}", fontsize=6, pad=2)
plt.show()




# ROUND 3

In [1]:
import torch
from torchvision import datasets, transforms
from matplotlib import pyplot as plt
from rich import print as prin

In [16]:

tfm = transforms.ToTensor()

dataset = datasets.MNIST(root="./data", train=True, transform=tfm, download=True)
loader = torch.utils.data.DataLoader(dataset, len(dataset), shuffle=False)
images, labels = next(iter(loader))

dataset_test = datasets.MNIST(root="./data", train=False, transform=tfm, download=True)
test_loader = torch.utils.data.DataLoader(dataset_test, len(dataset_test), shuffle=False)
images_test, labels_test = next(iter(test_loader))

In [19]:
X = images.view(len(dataset), -1)
X_mean = X.mean(0)
Xc = X - X_mean

U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)


